# Foundations Exercise: Professionally You (Tools + Evaluator)

This notebook stays within the concepts and tools taught in `1_foundations`:
- OpenAI-compatible API usage with provider routing (`OpenRouter`, `Groq`)
- Prompting with system + message history
- Tool calling with JSON tool schemas
- Tool-call loop (`while` loop + tool execution)
- Evaluator pattern and one retry on failure
- Gradio chat UI
- Context grounding from `summary.txt` and `linkedin.pdf`


## Concept Consolidation From `1_foundations`

1. `1_lab1`: API basics, `.env`, OpenAI-compatible chat completions, message formatting.
2. `2_lab2`: multi-model experimentation, model comparison, evaluator/judge prompts, provider routing.
3. `3_lab3`: personal-context chatbot (`summary` + `LinkedIn PDF`), quality evaluator, rerun after rejection.
4. `4_lab4`: tool definitions, function calling loop, real actions (notifications/recording), chatbot deployment pattern.

Applied here:
- Personal professional agent grounded in local profile data.
- Tool use to record leads, unanswered questions, and FAQ memory.
- Evaluator check before final response, with one controlled retry.


In [ ]:
# Imports (all from course scope)

import json
import os
from pathlib import Path

import gradio as gr
import requests
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from pypdf import PdfReader


In [ ]:
# Environment and clients

load_dotenv(override=True)

or_api_key = os.getenv("OR_API_KEY")
or_base_url = os.getenv("OR_BASE_URL")
groq_api_key = os.getenv("GROQ_API_KEY")
groq_base_url = os.getenv("GROQ_BASE_URL")

if not or_api_key or not or_base_url:
    raise ValueError("Missing OR_API_KEY or OR_BASE_URL in .env")
if not groq_api_key or not groq_base_url:
    raise ValueError("Missing GROQ_API_KEY or GROQ_BASE_URL in .env")

router = OpenAI(api_key=or_api_key, base_url=or_base_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_base_url)

ASSISTANT_MODEL = "openai/gpt-4o-mini"
EVALUATOR_MODEL = "openai/gpt-oss-120b"


In [ ]:
# Load your profile context (update these files for your own profile)

profile_dir = Path("../../me")
linkedin_pdf = profile_dir / "linkedin.pdf"
summary_txt = profile_dir / "summary.txt"

if not linkedin_pdf.exists() or not summary_txt.exists():
    raise FileNotFoundError(
        "Expected ../../me/linkedin.pdf and ../../me/summary.txt relative to this notebook."
    )

reader = PdfReader(str(linkedin_pdf))
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text + "\n"

summary = summary_txt.read_text(encoding="utf-8")
name = "Eddy Mmaitsimwale"


In [ ]:
# Notification + persistence helpers (same ideas as lab4)

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

storage_dir = Path("./runtime_data")
storage_dir.mkdir(exist_ok=True)

leads_file = storage_dir / "leads.json"
unknowns_file = storage_dir / "unknown_questions.json"
faq_file = storage_dir / "faq.json"


def _load_json(path, default):
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return default


def _save_json(path, data):
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def push(message):
    print(f"Push: {message}")
    if not pushover_user or not pushover_token:
        return {"sent": False, "reason": "Pushover not configured"}
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload, timeout=20)
    return {"sent": True}


In [ ]:
# Tool functions

def record_user_details(email, name="Name not provided", notes="not provided"):
    leads = _load_json(leads_file, [])
    leads.append({"email": email, "name": name, "notes": notes})
    _save_json(leads_file, leads)
    push(f"New lead: {name} ({email})")
    return {"recorded": "ok", "total_leads": len(leads)}


def record_unknown_question(question):
    unknowns = _load_json(unknowns_file, [])
    unknowns.append({"question": question})
    _save_json(unknowns_file, unknowns)
    push(f"Unknown question logged: {question[:120]}")
    return {"recorded": "ok", "total_unknown_questions": len(unknowns)}


def save_faq(question, answer):
    faq = _load_json(faq_file, {})
    faq[question.strip().lower()] = answer.strip()
    _save_json(faq_file, faq)
    return {"saved": "ok", "faq_size": len(faq)}


def lookup_faq(question):
    faq = _load_json(faq_file, {})
    key = question.strip().lower()
    if key in faq:
        return {"found": True, "answer": faq[key]}
    return {"found": False, "answer": ""}


In [ ]:
# Tool schemas

record_user_details_json = {
    "name": "record_user_details",
    "description": "Record a user's contact details when they show interest.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "User email"},
            "name": {"type": "string", "description": "User name"},
            "notes": {"type": "string", "description": "Extra context"},
        },
        "required": ["email"],
        "additionalProperties": False,
    },
}

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always call this when you cannot answer a question with confidence.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "Unanswered question"}
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

save_faq_json = {
    "name": "save_faq",
    "description": "Save a useful Q&A pair for future re-use.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string"},
            "answer": {"type": "string"},
        },
        "required": ["question", "answer"],
        "additionalProperties": False,
    },
}

lookup_faq_json = {
    "name": "lookup_faq",
    "description": "Look up previously saved answers in FAQ memory.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string"},
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
    {"type": "function", "function": save_faq_json},
    {"type": "function", "function": lookup_faq_json},
]

TOOL_REGISTRY = {
    "record_user_details": record_user_details,
    "record_unknown_question": record_unknown_question,
    "save_faq": save_faq,
    "lookup_faq": lookup_faq,
}


In [ ]:
# Tool dispatcher

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}")

        tool_fn = TOOL_REGISTRY.get(tool_name)
        result = tool_fn(**arguments) if tool_fn else {"error": "unknown tool"}

        results.append(
            {
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id,
            }
        )
    return results


In [ ]:
# Prompts

system_prompt = f"""You are acting as {name}. You are answering questions on {name}'s website,
particularly about {name}'s career, background, projects, and experience.

Your behavior rules:
1. Be professional, concise, and engaging.
2. Use only the provided profile context and conversation context.
3. If the user asks something you cannot answer confidently, call record_unknown_question.
4. If user intent is career/business contact, ask for their email and call record_user_details.
5. You may use lookup_faq and save_faq when useful.

## Summary:
{summary}

## LinkedIn Profile:
{linkedin}
"""


class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


evaluator_system_prompt = f"""You are an evaluator that checks if the assistant response is acceptable.
The assistant is representing {name} professionally on a personal website.
Reject answers that are clearly inaccurate, unprofessional, evasive, or ignore user intent.
Return strict JSON with keys: is_acceptable (bool), feedback (string).
"""


In [ ]:
# Evaluator + retry helpers

def evaluator_user_prompt(reply, message, history):
    prompt = f"Conversation history: {history}\n\n"
    prompt += f"Latest user message: {message}\n\n"
    prompt += f"Assistant reply: {reply}\n\n"
    prompt += "Evaluate this reply. Return only JSON."
    return prompt


def evaluate(reply, message, history) -> Evaluation:
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)},
    ]

    response = groq.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=messages,
        response_format={"type": "json_object"},
    )

    raw = response.choices[0].message.content
    try:
        parsed = json.loads(raw)
        return Evaluation(**parsed)
    except Exception:
        return Evaluation(is_acceptable=True, feedback="Evaluator parse failed; default pass.")


def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected by evaluator\n"
    updated_system_prompt += f"Rejected answer:\n{reply}\n\n"
    updated_system_prompt += f"Feedback:\n{feedback}\n"

    messages = [{"role": "system", "content": updated_system_prompt}] + history + [
        {"role": "user", "content": message}
    ]

    response = router.chat.completions.create(
        model=ASSISTANT_MODEL,
        messages=messages,
        tools=tools,
    )
    return response.choices[0].message.content


In [ ]:
# Main chat function: tool loop + evaluator + one retry

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [
        {"role": "user", "content": message}
    ]

    done = False
    final_reply = ""

    while not done:
        response = router.chat.completions.create(
            model=ASSISTANT_MODEL,
            messages=messages,
            tools=tools,
        )

        choice = response.choices[0]
        if choice.finish_reason == "tool_calls":
            llm_message = choice.message
            tool_calls = llm_message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(llm_message)
            messages.extend(results)
        else:
            final_reply = choice.message.content
            done = True

    evaluation = evaluate(final_reply, message, history)
    if not evaluation.is_acceptable:
        print("Evaluator rejected first reply. Rerunning once.")
        final_reply = rerun(final_reply, message, history, evaluation.feedback)

    return final_reply


In [ ]:
# Launch UI

gr.ChatInterface(chat, type="messages").launch()
